# 04 - Interpretability Analysis

Mechanistic interpretability of the trained Karatsuba transformer, following Nanda et al. (2023).

Key questions:
1. Do attention heads specialise for different parts of the Karatsuba algorithm?
2. Does the model use different loop iterations for different recursion levels?
3. Do embeddings learn Fourier-like structure (as in modular addition)?
4. What happens in the residual stream at each loop iteration?

**Run on Colab with any GPU. Analysis is lightweight.**

In [ ]:
# Cell 0: Setup
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx

# --- Repo setup ---
REPO_URL = "https://github.com/YOUR_USERNAME/karatsuba-transformers.git"  # TODO: update
REPO_DIR = "/content/karatsuba-transformers"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

!pip install -q -r {REPO_DIR}/requirements.txt
!pip install -q -e {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_BASE = '/content/drive/MyDrive/karatsuba_checkpoints'

from src.model.looped_transformer import KaratsubaTransformer
from src.data.dataset import MultiplicationDataset
from src.data.karatsuba_trace import generate_karatsuba_trace
from src.data.tokenizer import KaratsubaTokenizer

# Load Karatsuba model
with open(os.path.join(REPO_DIR, 'configs', '8bit_karatsuba.yaml'), 'r') as f:
    config = yaml.safe_load(f)

key = jrandom.PRNGKey(42)
model = KaratsubaTransformer(config['model'], key=key)
ckpt_path = os.path.join(CHECKPOINT_BASE, '8bit_karatsuba', 'best_model.eqx')
model = eqx.tree_deserialise_leaves(ckpt_path, model)

print(f"Model loaded from: {ckpt_path}")
print(f"d_model={config['model']['d_model']}, n_heads={config['model']['n_heads']}")
print("Setup complete.")

In [ ]:
# Cell 1: Attention pattern visualization
# Do attention heads specialise for different parts of the Karatsuba algorithm?

def extract_attention_patterns(model, tokens, positions, n_loops):
    """Run the model and capture attention weights at each loop iteration.

    Returns: attention_patterns of shape (n_loops, n_heads, seq_len, seq_len)
    """
    # Hook into the model to capture attention weights
    # This requires the model to support returning attention weights
    x = model.embed(tokens) + model.pos_encode(positions)

    all_attentions = []
    for loop_idx in range(n_loops):
        timestep = jnp.array(loop_idx)
        # Get attention weights from this loop iteration
        x, attn_weights = model.block.forward_with_attention(x, timestep)
        all_attentions.append(attn_weights)

    return jnp.stack(all_attentions)  # (n_loops, n_heads, seq_len, seq_len)


# Generate a sample 8-bit trace
tokenizer = KaratsubaTokenizer(config['data'])
x_val, y_val = 173, 211  # Example 8-bit numbers
tokens, positions, labels = tokenizer.encode_example(x_val, y_val, n_bits=8)
tokens_jax = jnp.array(tokens)
positions_jax = jnp.array(positions)

n_loops = 8
n_heads = config['model']['n_heads']

print(f"Input: {x_val} x {y_val} = {x_val * y_val}")
print(f"Sequence length: {len(tokens)}")
print(f"Extracting attention patterns for {n_loops} loops, {n_heads} heads...")

attn_patterns = extract_attention_patterns(model, tokens_jax, positions_jax, n_loops)
print(f"Attention shape: {attn_patterns.shape}")

# Plot attention patterns for each head at each loop iteration
fig, axes = plt.subplots(n_loops, n_heads, figsize=(3 * n_heads, 3 * n_loops))
if n_loops == 1:
    axes = axes[np.newaxis, :]
if n_heads == 1:
    axes = axes[:, np.newaxis]

for loop_idx in range(n_loops):
    for head_idx in range(n_heads):
        ax = axes[loop_idx][head_idx]
        attn = np.array(attn_patterns[loop_idx, head_idx])
        im = ax.imshow(attn, cmap='Blues', vmin=0, vmax=1, aspect='auto')
        if loop_idx == 0:
            ax.set_title(f'Head {head_idx}', fontsize=10)
        if head_idx == 0:
            ax.set_ylabel(f'Loop {loop_idx}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle(f'Attention Patterns: {x_val} x {y_val} ({len(tokens)} tokens)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'attention_patterns.png'), dpi=150, bbox_inches='tight')
plt.show()

# Identify head specialisation: compute attention entropy per head/loop
print("\nAttention Entropy (lower = more specialised):")
print(f"{'Loop':>6} {'Head':>6} {'Entropy':>10} {'Max Attn':>10}")
print("-" * 36)
for loop_idx in range(n_loops):
    for head_idx in range(n_heads):
        attn = np.array(attn_patterns[loop_idx, head_idx])
        # Mean entropy across query positions
        entropy = -np.sum(attn * np.log(attn + 1e-10), axis=-1).mean()
        max_attn = attn.max()
        print(f"{loop_idx:>6} {head_idx:>6} {entropy:>10.3f} {max_attn:>10.3f}")

In [ ]:
# Cell 2: Loop utilization analysis
# Does the model use different loop iterations for different purposes?
# Hypothesis: early loops handle decomposition, later loops handle recombination.

def extract_residual_norms(model, tokens, positions, n_loops):
    """Track the L2 norm of the residual stream change at each loop iteration.
    Large changes = the loop iteration is doing significant computation.
    Small changes = the model has converged / this iteration is unused.
    """
    x = model.embed(tokens) + model.pos_encode(positions)

    norms = []
    for loop_idx in range(n_loops):
        timestep = jnp.array(loop_idx)
        x_prev = x
        x = model.block(x, timestep)
        delta = x - x_prev
        norm = jnp.linalg.norm(delta, axis=-1).mean()  # mean over sequence
        norms.append(float(norm))

    return norms


# Compute loop utilisation for multiple examples at different bit widths
test_configs = [
    (8, 8, 'Training length (8-bit)'),
    (16, 12, 'Test: 16-bit'),
    (32, 16, 'Test: 32-bit'),
]

fig, axes = plt.subplots(1, len(test_configs), figsize=(6 * len(test_configs), 5))
if len(test_configs) == 1:
    axes = [axes]

np.random.seed(42)

for idx, (n_bits, n_loops, title) in enumerate(test_configs):
    ax = axes[idx]

    # Generate several examples and average
    all_norms = []
    n_examples = 20
    for _ in range(n_examples):
        x_val = np.random.randint(0, 2**n_bits)
        y_val = np.random.randint(0, 2**n_bits)
        tokens, positions, _ = tokenizer.encode_example(x_val, y_val, n_bits=n_bits)
        tokens_jax = jnp.array(tokens)
        positions_jax = jnp.array(positions)
        norms = extract_residual_norms(model, tokens_jax, positions_jax, n_loops)
        all_norms.append(norms)

    all_norms = np.array(all_norms)
    mean_norms = all_norms.mean(axis=0)
    std_norms = all_norms.std(axis=0)

    loops = np.arange(n_loops)
    ax.bar(loops, mean_norms, yerr=std_norms, color='steelblue', alpha=0.8, capsize=3)
    ax.set_xlabel('Loop Iteration', fontsize=12)
    ax.set_ylabel('Mean Residual Change (L2)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.set_xticks(loops)
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Loop Utilisation: Residual Stream Changes Per Iteration', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'loop_utilisation.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("- Large norms in early iterations: decomposition phase (splitting numbers)")
print("- Large norms in later iterations: recombination phase (combining sub-results)")
print("- Small norms: unused iterations (model has converged)")
print("- For longer inputs, expect more iterations to be utilised")

In [ ]:
# Cell 3: Embedding structure analysis (Fourier analysis)
# Following Nanda et al. (2023): does the model learn Fourier-like representations?
# For modular addition, models learn to embed numbers on circles in R^2.
# For multiplication, we might see similar structure.

# Extract the token embedding matrix
embed_weights = np.array(model.embed.weight)  # (vocab_size, d_model)
print(f"Embedding matrix shape: {embed_weights.shape}")

# For binary tokens (0 and 1), the embedding difference encodes the bit value
bit_0_embed = embed_weights[0]  # embedding of '0'
bit_1_embed = embed_weights[1]  # embedding of '1'
bit_diff = bit_1_embed - bit_0_embed

print(f"\nBit embedding difference norm: {np.linalg.norm(bit_diff):.4f}")
print(f"Bit 0 embedding norm: {np.linalg.norm(bit_0_embed):.4f}")
print(f"Bit 1 embedding norm: {np.linalg.norm(bit_1_embed):.4f}")

# Fourier analysis of the position embeddings
# For hierarchical positions, check if the bit_significance component
# learns a Fourier-like structure
if hasattr(model.pos_encode, 'bit_significance_embed'):
    pos_weights = np.array(model.pos_encode.bit_significance_embed.weight)
    print(f"\nPosition embedding shape: {pos_weights.shape}")

    # Compute the DFT of position embeddings
    n_positions = pos_weights.shape[0]
    fft_result = np.fft.fft(pos_weights, axis=0)
    fft_magnitude = np.abs(fft_result)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Raw position embeddings (first 16 dimensions)
    ax = axes[0]
    im = ax.imshow(pos_weights[:, :16].T, cmap='RdBu_r', aspect='auto',
                   norm=TwoSlopeNorm(0))
    ax.set_xlabel('Bit Significance', fontsize=11)
    ax.set_ylabel('Embedding Dimension', fontsize=11)
    ax.set_title('Position Embeddings (first 16 dims)', fontsize=12)
    plt.colorbar(im, ax=ax)

    # FFT magnitude
    ax = axes[1]
    im = ax.imshow(fft_magnitude[:n_positions//2, :16].T, cmap='hot', aspect='auto')
    ax.set_xlabel('Frequency', fontsize=11)
    ax.set_ylabel('Embedding Dimension', fontsize=11)
    ax.set_title('FFT of Position Embeddings', fontsize=12)
    plt.colorbar(im, ax=ax)

    # PCA of position embeddings
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    pos_2d = pca.fit_transform(pos_weights)

    ax = axes[2]
    scatter = ax.scatter(pos_2d[:, 0], pos_2d[:, 1], c=range(n_positions),
                         cmap='viridis', s=100, edgecolors='black', linewidth=0.5)
    for i in range(n_positions):
        ax.annotate(str(i), (pos_2d[i, 0], pos_2d[i, 1]), fontsize=8, ha='center')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontsize=11)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontsize=11)
    ax.set_title('PCA of Position Embeddings', fontsize=12)
    plt.colorbar(scatter, ax=ax, label='Bit Position')
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(os.path.join(CHECKPOINT_BASE, 'embedding_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nPCA explained variance: {pca.explained_variance_ratio_[:5]}")
    print("If embeddings lie on a circle/spiral, the model has learned a Fourier-like representation.")
else:
    print("Position encoding does not have separate bit_significance embedding. Skipping Fourier analysis.")
    print("Check model.pos_encode attributes to find the right embedding to analyse.")

In [ ]:
# Cell 4: Residual stream analysis
# Track information flow through the residual stream across loop iterations.
# Can we identify when the model transitions from decomposition to recombination?

def extract_residual_stream(model, tokens, positions, n_loops):
    """Extract the full residual stream at each loop iteration.
    Returns: list of arrays, each of shape (seq_len, d_model)
    """
    x = model.embed(tokens) + model.pos_encode(positions)
    streams = [np.array(x)]  # initial embedding

    for loop_idx in range(n_loops):
        timestep = jnp.array(loop_idx)
        x = model.block(x, timestep)
        streams.append(np.array(x))

    return streams


# Extract residual stream for an example
x_val, y_val = 173, 211
tokens, positions, labels = tokenizer.encode_example(x_val, y_val, n_bits=8)
n_loops = 8

streams = extract_residual_stream(
    model, jnp.array(tokens), jnp.array(positions), n_loops
)

print(f"Residual stream: {len(streams)} snapshots, each shape {streams[0].shape}")

# 1. Cosine similarity between consecutive loop iterations
cos_sims = []
for i in range(1, len(streams)):
    # Flatten and compute cosine similarity
    a = streams[i].flatten()
    b = streams[i-1].flatten()
    cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)
    cos_sims.append(cos_sim)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cosine similarity between consecutive iterations
ax = axes[0, 0]
ax.plot(range(1, len(cos_sims) + 1), cos_sims, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('Loop Iteration', fontsize=11)
ax.set_ylabel('Cosine Similarity with Previous', fontsize=11)
ax.set_title('Residual Stream Convergence', fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.5, 1.05])

# Plot 2: PCA of residual stream across iterations (averaged over sequence)
from sklearn.decomposition import PCA
stream_means = np.array([s.mean(axis=0) for s in streams])  # (n_loops+1, d_model)
pca = PCA(n_components=2)
stream_2d = pca.fit_transform(stream_means)

ax = axes[0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, len(streams)))
for i in range(len(streams)):
    label = 'Embed' if i == 0 else f'Loop {i}'
    ax.scatter(stream_2d[i, 0], stream_2d[i, 1], c=[colors[i]], s=100,
               edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(label, (stream_2d[i, 0], stream_2d[i, 1]),
                fontsize=8, ha='left', va='bottom')
# Draw path
ax.plot(stream_2d[:, 0], stream_2d[:, 1], '--', color='gray', alpha=0.5, linewidth=1)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax.set_title('Residual Stream Trajectory (PCA)', fontsize=12)
ax.grid(True, alpha=0.2)

# Plot 3: Heatmap of residual stream norms per position per iteration
norms_by_pos = np.array([[np.linalg.norm(s[j]) for j in range(s.shape[0])] for s in streams])

ax = axes[1, 0]
im = ax.imshow(norms_by_pos, cmap='hot', aspect='auto')
ax.set_xlabel('Token Position', fontsize=11)
ax.set_ylabel('Loop Iteration (0=embed)', fontsize=11)
ax.set_title('Residual Stream Norms', fontsize=12)
plt.colorbar(im, ax=ax, label='L2 Norm')

# Plot 4: Difference norms (where is computation happening?)
diff_norms = np.array([
    [np.linalg.norm(streams[i][j] - streams[i-1][j]) for j in range(streams[i].shape[0])]
    for i in range(1, len(streams))
])

ax = axes[1, 1]
im = ax.imshow(diff_norms, cmap='hot', aspect='auto')
ax.set_xlabel('Token Position', fontsize=11)
ax.set_ylabel('Loop Iteration', fontsize=11)
ax.set_title('Residual Stream Changes (where computation happens)', fontsize=12)
plt.colorbar(im, ax=ax, label='Change L2 Norm')

plt.suptitle(f'Residual Stream Analysis: {x_val} x {y_val} = {x_val * y_val}', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_BASE, 'residual_stream_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observations to look for:")
print("- Phase transition in cosine similarity: marks the switch from decompose to recombine")
print("- Residual trajectory: spiral = progressive refinement; L-shape = two distinct phases")
print("- Change norms: hot spots show which token positions are being updated at each loop")
print("  (expect activity at split boundaries, then at combine positions)")